<a href="https://colab.research.google.com/github/Derfed001/ai_voiceText_clone/blob/main/voice_twin_server_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VoiceTwin server (free GPU)

This runs the "brain" that your phone app talks to. Everything the agents learn is saved in your **Google Drive**, so it survives Colab resets.

**Before you start**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Key icon (Secrets) on the left: add `GEMINI_API_KEY` (free, from aistudio.google.com/apikey) and `VT_TOKEN` (any password you invent). Turn on notebook access for both.
3. Run the cells in order. The last cell prints a **URL**. Paste it, plus your token, into the app's **Brain** tab.

The URL changes every time you start this notebook. Keep this tab open while you use the app. Colab may stop idle sessions.

## 1. Save learning to your Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['VT_DATA'] = '/content/drive/MyDrive/voicetwin_data'
os.makedirs(os.environ['VT_DATA'], exist_ok=True)
print('Data folder:', os.environ['VT_DATA'])

Mounted at /content/drive
Data folder: /content/drive/MyDrive/voicetwin_data


## 2. Install (2 to 3 minutes)

In [2]:
!pip install -q fastapi uvicorn python-multipart requests numpy coqui-tts "transformers==4.57.6"
!ffmpeg -version | head -1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.3/639.3 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the pac

## 3. Server code

In [3]:
%%writefile server.py
"""
VoiceTwin server: the "brain" that keeps learning every time you upload.

What learns automatically
  TEXT agent : every upload / correction goes into a library. A background learner folds new material into
               the style guide (incrementally, keeping your manual edits), and every chat reply is grounded in
               lines you really wrote on similar topics plus your past corrections.
  VOICE agent: every clip is converted, quality-scored (noise, clipping, length) and the 3 best clips are
               automatically chosen as the reference voice. Upload a better clip and it takes over by itself.

Run locally:   pip install fastapi uvicorn python-multipart requests numpy
               (voice replies also need: pip install coqui-tts "transformers==4.57.6" and ffmpeg installed)
               set GEMINI_API_KEY=...   set VT_TOKEN=pick-a-password
               uvicorn server:app --host 0.0.0.0 --port 8000
Or run it on Colab (see voice_twin_server_colab.ipynb).
"""
import io
import json
import math
import os
import re
import secrets
import subprocess
import threading
import time
import wave
import zipfile
import hashlib
from collections import Counter
from contextlib import asynccontextmanager
from pathlib import Path
from typing import List

import numpy as np
import requests
from fastapi import BackgroundTasks, Depends, FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import Response
from pydantic import BaseModel

DATA = Path(os.environ.get("VT_DATA", "./vt_data")).resolve()
TEXT, RAW, REFS, STYLES, CACHE = (DATA / n for n in ("text", "audio_raw", "refs", "styles", "cache"))
for _d in (TEXT, RAW, REFS, STYLES, CACHE):
    _d.mkdir(parents=True, exist_ok=True)
FB_FILE = DATA / "feedback.jsonl"
STATE_FILE = DATA / "state.json"
ANALYSIS_FILE = DATA / "analysis.json"

TOKEN = os.environ.get("VT_TOKEN") or secrets.token_urlsafe(12)
GEMINI_KEY = os.environ.get("GEMINI_API_KEY", "")
MODELS = os.environ.get("VT_MODELS", "gemini-3.5-flash,gemini-3.8-flash,gemini-3.1-flash-lite").split(",")
LANG = os.environ.get("VT_LANG", "en")
MIN_GAP_S = int(os.environ.get("VT_LEARN_GAP", "120"))   # min seconds between style updates
NEW_CHARS = int(os.environ.get("VT_LEARN_CHARS", "1500"))  # new text needed to trigger an update
NEW_FB = int(os.environ.get("VT_LEARN_FB", "5"))           # new corrections needed to trigger an update

_state_lock = threading.RLock()
_learn_lock = threading.Lock()
_tts_lock = threading.Lock()
_learning = False
_tts = None


def auth(x_token: str = Header(default="")):
    if not secrets.compare_digest(x_token.encode(), TOKEN.encode()):
        raise HTTPException(401, "Wrong token")


# ----------------------------------------------------------------- state
def load_state() -> dict:
    with _state_lock:
        try:
            return json.loads(STATE_FILE.read_text())
        except Exception:
            return {}


def save_state(s: dict):
    with _state_lock:
        STATE_FILE.write_text(json.dumps(s))


def read_fb() -> List[dict]:
    if not FB_FILE.exists():
        return []
    out = []
    for line in FB_FILE.read_text(encoding="utf-8").splitlines():
        try:
            o = json.loads(line)
            if o.get("prompt") is not None and o.get("ideal"):
                out.append(o)
        except Exception:
            pass
    return out


def current_style() -> str:
    p = STYLES / "current.md"
    return p.read_text(encoding="utf-8") if p.exists() else ""


# ----------------------------------------------------------------- text cleaning
WA_LINE = re.compile(
    r"^\[?(\d{1,2}/\d{1,2}/\d{2,4}),? (\d{1,2}:\d{2})(?::\d{2})?[\s\u202f\u00a0]?([APap][Mm])?\]?\s*-?\s*([^:]+?): (.*)$"
)
DATE_START = re.compile(r"^\[?\d{1,2}/\d{1,2}/\d{2,4},? \d{1,2}:\d{2}")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
PHONE_RE = re.compile(r"(?<!\d)\+?\d[\d\s\-]{8,}\d(?!\d)")
MENTION_RE = re.compile(r"@[^\s@]+")
SKIP = {"<Media omitted>", "This message was deleted", "You deleted this message", "null"}


def clean(s: str, max_len: int = 800):
    s = s.replace("<This message was edited>", "").strip()
    if not s or s in SKIP or "(file attached)" in s or len(s) > max_len:
        return None
    s = URL_RE.sub("[link]", s)
    s = EMAIL_RE.sub("[email]", s)
    s = PHONE_RE.sub("[number]", s)
    return MENTION_RE.sub("@name", s)


def extract_messages(raw: str, wa_name: str):
    """Returns a list of cleaned messages, or None if this is a WhatsApp export and no name was given."""
    lines = raw.replace("\r", "").split("\n")
    is_wa = any(WA_LINE.match(l) for l in lines[:30])
    kept: List[str] = []
    max_len = 800 if is_wa else 3000  # long WhatsApp messages are usually pasted/forwarded; plain text is your own writing
    if is_wa:
        me = wa_name.strip().lower()
        if not me:
            return None
        cur = None
        for line in lines:
            m = WA_LINE.match(line)
            if m:
                if cur is not None and cur.strip():
                    kept.append(cur)
                cur = m.group(5) if m.group(4).strip().lower() == me else None
            elif DATE_START.match(line):
                if cur is not None and cur.strip():
                    kept.append(cur)
                cur = None
            elif cur is not None:
                cur += "\n" + line
        if cur is not None and cur.strip():
            kept.append(cur)
    else:
        kept = lines
    return [c for c in (clean(k, max_len) for k in kept) if c]


# ----------------------------------------------------------------- Gemini
def gemini(system: str, turns: List[dict]) -> str:
    if not GEMINI_KEY:
        raise RuntimeError("GEMINI_API_KEY is not set on the server")
    last = ""
    for m in MODELS:
        try:
            body = {
                "system_instruction": {"parts": [{"text": system}]},
                "contents": [{"role": t["role"], "parts": [{"text": t["text"]}]} for t in turns],
            }
            r = requests.post(
                f"https://generativelanguage.googleapis.com/v1beta/models/{m}:generateContent",
                headers={"x-goog-api-key": GEMINI_KEY, "Content-Type": "application/json"},
                json=body,
                timeout=120,
            )
            if r.status_code == 200:
                parts = r.json()["candidates"][0]["content"]["parts"]
                return "".join(p.get("text", "") for p in parts)
            last = f"{m}: HTTP {r.status_code}"
        except Exception as e:  # noqa
            last = f"{m}: {e}"
    raise RuntimeError(f"Gemini failed ({last})")


# ----------------------------------------------------------------- retrieval (BM25, no extra libraries)
def tok(s: str):
    return re.findall(r"\w+", s.lower())


class BM25:
    def __init__(self, docs: List[str]):
        self.raw = docs
        self.docs = [tok(d) for d in docs]
        self.n = len(docs)
        self.avg = sum(len(d) for d in self.docs) / max(1, self.n)
        self.tf = [Counter(d) for d in self.docs]
        self.df = Counter()
        for d in self.docs:
            for t in set(d):
                self.df[t] += 1

    def top(self, q: str, k: int = 8):
        qt = set(tok(q))
        scored = []
        for i, tf in enumerate(self.tf):
            s = 0.0
            ln = len(self.docs[i]) or 1
            for t in qt:
                f = tf.get(t)
                if f:
                    idf = math.log(1 + (self.n - self.df[t] + 0.5) / (self.df[t] + 0.5))
                    s += idf * f * 2.2 / (f + 1.2 * (0.25 + 0.75 * ln / (self.avg or 1)))
            if s > 0:
                scored.append((s, i))
        scored.sort(reverse=True)
        return [self.raw[i] for _, i in scored[:k]]


_index = {"sig": None, "lines": None, "pairs": None, "fb": []}


def get_index():
    files = sorted(TEXT.glob("*.txt"))
    fb = read_fb()
    sig = (len(files), sum(f.stat().st_size for f in files), len(fb))
    if _index["sig"] != sig:
        lines = []
        for f in files:
            for l in f.read_text(encoding="utf-8", errors="ignore").split("\n"):
                l = l.strip()
                if 12 <= len(l) <= 400:
                    lines.append(l)
        _index.update(
            sig=sig,
            lines=BM25(lines) if lines else None,
            pairs=BM25([o["prompt"] for o in fb]) if fb else None,
            fb=fb,
        )
    return _index


# ----------------------------------------------------------------- the text agent: persona + learner
def persona(style: str, lines: List[str], pairs: List[dict], spoken: bool) -> str:
    s = (
        "You are a digital twin of the person whose style guide follows. Reply exactly as they would: same tone, rhythm, "
        "vocabulary and habits, switching between their casual and professional registers to fit the situation. "
        "Keep replies short and conversational. Never invent personal facts, numbers or events about them. "
        "Never claim to be a different person, and if sincerely asked, admit you are an AI imitating their style.\n"
    )
    if spoken:
        s += "This reply will be spoken aloud: no emoji, no markdown, 1 to 3 short sentences.\n"
    s += f"\n<style_guide>\n{style or '(none yet, use a casual, direct tone)'}\n</style_guide>\n"
    if lines:
        s += "\nThings they really wrote on similar topics (match this voice):\n" + "\n".join(f"- {l[:300]}" for l in lines) + "\n"
    if pairs:
        s += "\nReal examples of how they replied (follow these closely):\n"
        for o in pairs:
            s += f"Someone said: {o['prompt'][:300]}\nThey replied: {o['ideal'][:300]}\n\n"
    return s


def learn_if_needed(force: bool = False):
    """Folds new texts and corrections into the style guide. Safe to call any time."""
    global _learning
    if not _learn_lock.acquire(blocking=False):
        return
    try:
        if not GEMINI_KEY:
            return
        st = load_state()
        now = time.time()
        if not force and now - st.get("last_learn", 0) < MIN_GAP_S:
            return
        seen = set(st.get("seen_text", []))
        all_files = sorted(TEXT.glob("*.txt"))
        new_files = [f for f in all_files if f.name not in seen]
        fb = read_fb()
        new_fb = fb[st.get("fb_seen", 0):]
        new_text = "\n".join(f.read_text(encoding="utf-8", errors="ignore") for f in new_files)
        have = bool(current_style())
        if not force:
            if have and len(new_text) < NEW_CHARS and len(new_fb) < NEW_FB:
                return
            if not have and not new_text and not new_fb:
                return
        if not new_text and not new_fb and have:
            return
        _learning = True
        fb_txt = "\n".join(f"Someone said: {o['prompt'][:300]}\nThey would reply: {o['ideal'][:300]}\n" for o in (fb if not have else new_fb))
        if have:
            material = new_text[:60000]
            prompt = (
                "Below is the CURRENT style guide for one person, followed by NEW material (writing samples and corrections "
                "showing how they really wanted to reply). Update the guide: keep everything that is still true (including "
                "anything that looks hand-edited), refine or correct points using the new evidence, add new patterns, and "
                "treat corrections as the strongest evidence. Keep the same structure and keep it prompt-ready. "
                "Never include verbatim lines about health, money, relationships, other people's names or private details, "
                "and never invent facts about the person. Output ONLY the full updated guide.\n\n"
                f"<current_guide>\n{current_style()}\n</current_guide>\n\n<new_samples>\n{material}\n</new_samples>\n\n"
                f"<new_corrections>\n{fb_txt}\n</new_corrections>"
            )
        else:
            corpus = ""
            for f in all_files:
                if len(corpus) > 120000:
                    break
                corpus += f.read_text(encoding="utf-8", errors="ignore") + "\n\n---\n\n"
            prompt = (
                "Below are writing samples from one person. Write a prompt-ready style guide another writer could follow to imitate them. "
                "If they write differently in different situations (e.g. casual with friends vs professional with clients), describe each "
                "register separately and say when to use each. Cover tone, sentence length and rhythm, vocabulary and favourite phrases "
                "(including any Swahili or Sheng; give meanings only when you are sure), punctuation and capitalisation, emoji use, openings "
                "and closings, humour, language mixing, and what they avoid. Add a short section of rules for replies that will be spoken "
                "aloud (no emoji, short sentences). Finish with 6 short verbatim example lines that show their voice, none of which mention "
                "health, money, relationships, other people's names or private details. Never invent facts about the person.\n\n"
                f"<samples>\n{corpus}\n</samples>\n\n<corrections>\n{fb_txt}\n</corrections>"
            )
        guide = gemini("You are a stylometry expert. Produce precise, practical style guides.", [{"role": "user", "text": prompt}])
        ver = st.get("style_version", 0) + 1
        (STYLES / f"style_v{ver}.md").write_text(guide, encoding="utf-8")
        (STYLES / "current.md").write_text(guide, encoding="utf-8")
        st.update(style_version=ver, last_learn=time.time(), seen_text=[f.name for f in all_files], fb_seen=len(fb))
        save_state(st)
        print(f"[learner] style guide updated to v{ver}")
    except Exception as e:  # noqa
        print("[learner] failed:", e)
    finally:
        _learning = False
        _learn_lock.release()


def _loop():
    while True:
        time.sleep(300)
        try:
            learn_if_needed()
        except Exception as e:  # noqa
            print("[loop]", e)


# ----------------------------------------------------------------- the voice agent: analysis + automatic reference selection
def read_wav(path: Path):
    with wave.open(str(path), "rb") as w:
        sr = w.getframerate()
        ch = w.getnchannels()
        x = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16).astype(np.float32)
    if ch > 1:
        x = x.reshape(-1, ch).mean(axis=1)
    return x, sr


def analyze(path: Path) -> dict:
    x, sr = read_wav(path)
    dur = len(x) / sr if sr else 0.0
    if len(x) < sr // 2:
        return {"seconds": round(dur, 1), "snr_db": 0.0, "level_db": -99.0, "clip_pct": 0.0, "score": -50.0, "notes": ["too short"]}
    win = max(1, int(sr * 0.05))
    n = len(x) // win
    frames = x[: n * win].reshape(n, win)
    rms = np.sqrt((frames ** 2).mean(axis=1)) / 32768.0
    db = 20 * np.log10(rms + 1e-9)
    noise = float(np.percentile(db, 10))
    speech = float(np.percentile(db, 90))
    snr = speech - noise
    clip = float((np.abs(x) >= 32000).mean())
    voiced = float((db > noise + 10).mean())
    notes = []
    score = min(snr, 40.0) + 10 * voiced - clip * 500
    if dur < 6:
        score -= 15
        notes.append("short (aim for 10 to 30 s)")
    if speech < -35:
        score -= 10
        notes.append("very quiet, speak closer to the mic")
    if snr < 15:
        notes.append("background noise, try a quieter room")
    if clip > 0.002:
        notes.append("clipping, speak a little further from the mic")
    return {"seconds": round(dur, 1), "snr_db": round(snr, 1), "level_db": round(speech, 1),
            "clip_pct": round(clip * 100, 2), "score": round(score, 1), "notes": notes}


def select_refs() -> List[dict]:
    """Scores every clip and keeps the best 3 as the reference voice."""
    try:
        cache = json.loads(ANALYSIS_FILE.read_text())
    except Exception:
        cache = {}
    clips = sorted(RAW.glob("clip_*.wav"))
    for c in clips:
        if c.name not in cache:
            cache[c.name] = analyze(c)
    ANALYSIS_FILE.write_text(json.dumps(cache))
    ranked = sorted(clips, key=lambda c: cache[c.name]["score"], reverse=True)[:3]
    for old in REFS.glob("*.wav"):
        old.unlink()
    for i, c in enumerate(ranked):
        (REFS / f"ref{i + 1}_{c.name}").write_bytes(c.read_bytes())
    return [{"name": c.name, **cache[c.name]} for c in ranked]


def get_tts():
    global _tts
    if _tts is None:
        os.environ["COQUI_TOS_AGREED"] = "1"  # running this means you accept Coqui's XTTS-v2 (non-commercial) licence
        try:
            import torch
            from TTS.api import TTS
        except Exception as e:  # noqa
            raise HTTPException(501, f"Voice model not installed on the server: {e}")
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        _tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(dev)
    return _tts


# ----------------------------------------------------------------- API
@asynccontextmanager
async def lifespan(app: FastAPI):
    threading.Thread(target=_loop, daemon=True).start()
    if not os.environ.get("VT_TOKEN"):
        print(f"[server] VT_TOKEN not set, generated one for this run: {TOKEN}")
    yield


app = FastAPI(title="VoiceTwin", lifespan=lifespan)


@app.get("/health")
def health():
    return {"ok": True}


@app.get("/status")
def status(_=Depends(auth)):
    st = load_state()
    files = list(TEXT.glob("*.txt"))
    try:
        an = json.loads(ANALYSIS_FILE.read_text())
    except Exception:
        an = {}
    refs = [{"name": r.name, **an.get(r.name.split("_", 1)[-1], {})} for r in sorted(REFS.glob("*.wav"))]
    return {
        "text_files": len(files),
        "text_chars": sum(f.stat().st_size for f in files),
        "audio_clips": len(list(RAW.glob("clip_*.wav"))),
        "ref_clips": refs,
        "corrections": len(read_fb()),
        "style_version": st.get("style_version", 0),
        "last_learn": st.get("last_learn", 0),
        "learning": _learning,
        "gemini_key_set": bool(GEMINI_KEY),
        "voice_model_loaded": _tts is not None,
    }


@app.post("/upload/text")
def upload_text(
    bg: BackgroundTasks,
    text: str = Form(""),
    wa_name: str = Form(""),
    files: List[UploadFile] = File(default=[]),
    _=Depends(auth),
):
    raws = []
    if text.strip():
        raws.append(text)
    for f in files:
        b = f.file.read()
        if b[:2] == b"PK":
            with zipfile.ZipFile(io.BytesIO(b)) as z:
                for n in z.namelist():
                    if n.lower().endswith(".txt"):
                        raws.append(z.read(n).decode("utf-8", errors="ignore"))
        else:
            raws.append(b.decode("utf-8", errors="ignore"))
    kept: List[str] = []
    for r in raws:
        m = extract_messages(r, wa_name)
        if m is None:
            raise HTTPException(400, "This looks like a WhatsApp export. Enter your name exactly as it appears in the chat, then try again.")
        kept += m
    if not kept:
        return {"kept": 0, "note": "Nothing usable found."}
    (TEXT / f"t_{int(time.time() * 1000)}.txt").write_text("\n".join(kept), encoding="utf-8")
    bg.add_task(learn_if_needed)
    return {"kept": len(kept), "note": "Saved. The text agent will fold this into your style automatically."}


@app.post("/upload/audio")
def upload_audio(bg: BackgroundTasks, file: UploadFile = File(...), _=Depends(auth)):
    data = file.file.read()
    if len(data) < 2000:
        raise HTTPException(400, "That file is too small to be a recording.")
    ts = int(time.time() * 1000)
    raw = RAW / f"raw_{ts}{Path(file.filename or 'x.wav').suffix or '.wav'}"
    raw.write_bytes(data)
    wav = RAW / f"clip_{ts}.wav"
    try:
        r = subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw), "-ac", "1", "-ar", "24000", "-t", "30", str(wav)],
            capture_output=True,
        )
    except FileNotFoundError:
        raw.unlink(missing_ok=True)
        raise HTTPException(500, "ffmpeg is not installed on the server")
    if r.returncode != 0 or not wav.exists():
        raw.unlink(missing_ok=True)
        raise HTTPException(400, "Could not read that audio: " + r.stderr.decode(errors="ignore")[:200])
    raw.unlink(missing_ok=True)  # keep only the clean 24 kHz mono clip
    info = analyze(wav)
    refs = select_refs()
    return {"clip": wav.name, **info, "in_use": any(x["name"] == wav.name for x in refs), "refs": refs}


class FbIn(BaseModel):
    prompt: str
    ideal: str


@app.post("/feedback")
def feedback(body: FbIn, bg: BackgroundTasks, _=Depends(auth)):
    if not body.ideal.strip():
        raise HTTPException(400, "empty")
    with _state_lock:
        with FB_FILE.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"prompt": body.prompt, "ideal": body.ideal.strip(), "t": time.time()}) + "\n")
    bg.add_task(learn_if_needed)
    return {"ok": True}


class ChatIn(BaseModel):
    message: str
    history: List[dict] = []
    spoken: bool = False


@app.post("/chat")
def chat(body: ChatIn, bg: BackgroundTasks, _=Depends(auth)):
    idx = get_index()
    lines = idx["lines"].top(body.message, 8) if idx["lines"] else []
    pairs = []
    if idx["pairs"]:
        hits = idx["pairs"].top(body.message, 5)
        pairs = [o for o in idx["fb"] if o["prompt"] in hits][:5]
    recent = idx["fb"][-5:]
    for o in recent:
        if o not in pairs:
            pairs.append(o)
    system = persona(current_style(), lines, pairs[:10], body.spoken)
    turns = [{"role": "model" if h.get("role") == "model" else "user", "text": str(h.get("text", ""))} for h in body.history[-12:]]
    turns.append({"role": "user", "text": body.message})
    while turns and turns[0]["role"] == "model":
        turns.pop(0)
    try:
        reply = gemini(system, turns)
    except Exception as e:  # noqa
        raise HTTPException(502, str(e))
    bg.add_task(learn_if_needed)
    return {"reply": reply.strip()}


class SpeakIn(BaseModel):
    text: str


@app.post("/speak")
def speak(body: SpeakIn, _=Depends(auth)):
    refs = sorted(REFS.glob("*.wav"))
    if not refs:
        raise HTTPException(400, "Upload at least one voice clip first.")
    sig = hashlib.sha1((body.text + "|".join(f"{r.name}{r.stat().st_size}" for r in refs)).encode()).hexdigest()
    out = CACHE / f"{sig}.wav"
    if not out.exists():
        tts = get_tts()
        with _tts_lock:
            tts.tts_to_file(text=body.text, speaker_wav=[str(r) for r in refs], language=LANG, file_path=str(out))
    return Response(out.read_bytes(), media_type="audio/wav")


class StyleIn(BaseModel):
    text: str


@app.get("/style")
def get_style(_=Depends(auth)):
    return {"version": load_state().get("style_version", 0), "text": current_style()}


@app.put("/style")
def put_style(body: StyleIn, _=Depends(auth)):
    st = load_state()
    ver = st.get("style_version", 0) + 1
    (STYLES / f"style_v{ver}.md").write_text(body.text, encoding="utf-8")
    (STYLES / "current.md").write_text(body.text, encoding="utf-8")
    st["style_version"] = ver
    save_state(st)
    return {"version": ver}


@app.post("/learn")
def learn_now(bg: BackgroundTasks, _=Depends(auth)):
    bg.add_task(learn_if_needed, True)
    return {"ok": True}

Writing server.py


## 4. Start the server and open a free public tunnel

In [4]:
import os, re, subprocess, time, secrets, urllib.request
from google.colab import userdata

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
try:
    os.environ['VT_TOKEN'] = userdata.get('VT_TOKEN')
except Exception:
    os.environ['VT_TOKEN'] = secrets.token_urlsafe(10)
    print('No VT_TOKEN secret found, generated one for this session.')

log = open('/content/server.log', 'w')
srv = subprocess.Popen(['python', '-m', 'uvicorn', 'server:app', '--host', '127.0.0.1', '--port', '8000'],
                       stdout=log, stderr=subprocess.STDOUT, env=os.environ)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    print(open('/content/server.log').read()[-2000:])
    raise SystemExit('Server did not start, see the log above.')

if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 60 and not url:
    line = tun.stdout.readline()
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line or '')
    if m:
        url = m.group(0)
print('\nPaste these into the app (Brain tab):')
print('  Server URL:', url)
print('  Token     :', os.environ['VT_TOKEN'])

No VT_TOKEN secret found, generated one for this session.

Paste these into the app (Brain tab):
  Server URL: https://can-designer-lips-interference.trycloudflare.com
  Token     : pZfIVmmnFrbKnQ


## Troubleshooting
If the app can't connect, run the next cell to see the server log.

In [5]:
print(open('/content/server.log').read()[-3000:])

INFO:     Started server process [2098]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:39436 - "GET /health HTTP/1.1" 200 OK

